# 06 – S7: ketahanan terhadap pergeseran resolusi (CP6)

Latih (generator dan klasifikator) hanya pada pasien resolusi mayoritas; uji gabungan resolusi lain (L5). Metode: S1 dan konfigurasi terbaik S5 dan S6 dari validasi. **Catatan interpretasi (PRD §6):** proporsi ganas berbeda antar resolusi, sehingga ini mengukur pergeseran akuisisi yang bercampur dengan perbedaan populasi, bukan efek resolusi murni.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.

from src.jobs import launch_scenario, launch_gan_chain
from src.evaluate import evaluate_scenario, s7_gap

cfg7 = load_merged('configs/s7_s1.yaml')   # splits_name = s7
s7 = json.loads(P(cfg, 'splits', 'splits_s7.json').read_text())
print('latih:', s7['major_resolution'], '| uji:', s7['test_resolutions'], '| n_repeats:', len(s7['repeats']))

## 1. S7_S1

In [ ]:
launch_scenario('configs/s7_s1.yaml', 'src.train', REPS, None, DEVICES)

## 2. S7_S5 dan S7_S6 (GAN dilatih hanya pada pasien resolusi mayoritas; rasio = konfigurasi terbaik dari validasi)

In [ ]:
launch_gan_chain('configs/s7_s5.yaml', ratios=[None], reps=REPS, devices=DEVICES)
launch_gan_chain('configs/s7_s6.yaml', ratios=[None], reps=REPS, devices=DEVICES)

## 3. Hasil dengan interval kepercayaan, dan selisih dalam-distribusi vs pergeseran (**eksploratif**)

In [ ]:
ALLOW_PARTIAL = REPS is not None
for sc, path in (('S7_S1', 'configs/s7_s1.yaml'), ('S7_S5', 'configs/s7_s5.yaml'), ('S7_S6', 'configs/s7_s6.yaml')):
    display(evaluate_scenario(load_merged(path), sc, ALLOW_PARTIAL)['table'])
for m in ('S1', 'S5', 'S6'):
    display(s7_gap(cfg7, m, f'S7_{m}'))

## 4. Pemeriksaan checkpoint CP6 dan `manifest.json` (sel terakhir)

In [ ]:
from src.checkpoint import finalize_cp6
report = finalize_cp6(cfg7)
assert report['checkpoint'] == 'CP6'